In [2]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import create_preprocessor

FE_DF_PATH = Path("../data/processed/openaq_feature_engineered.parquet")
fe_df = pd.read_parquet(FE_DF_PATH)

In [3]:
# look at the df
fe_df.head()

,Country Code,City,Location,Pollutant,Source Name,Unit,Country Label,Val Lag 1 Day,Year,Month,Day,Month Sin,Month Cos,Latitude,Longitude,Has Coordinates,Has City,Pollutant Danger,Value
0,CN,Unknown,十里堡居委,PM2.5,ChinaAQIData,µg/m³,China,NaN,2021,8,9,-0.866025,-5.000000e-01,35.0215,118.3564,1,0,Most Dangerous,39.000000
1,ZA,City of Cape Town,Cape Point-NAQI,NO,South Africa,ppm,South Africa,NaN,2023,5,30,0.500000,-8.660254e-01,-34.3533,18.4898,1,1,Highly Harmful,0.000537
2,IT,RETE REGIONALE UMBRIA,NET.IT252A,SO2,EEA Italy,µg/m³,Italy,1.3,2024,3,9,1.000000,6.123234e-17,43.1031,12.3661,1,1,Highly Harmful,1.300000
3,CN,Unknown,太平,PM10,ChinaAQIData,µg/m³,China,NaN,2021,8,9,-0.866025,-5.000000e-01,41.1442,123.0485,1,0,Moderate,41.000000
4,US,Houston,Galveston 99th St. C1034/A320/X183,O3,Texas,ppm,United States,NaN,2016,3,6,1.000000,6.123234e-17,29.2545,-94.8613,1,1,Highly Harmful,0.062000


In [4]:
# create the data
X = fe_df.drop("Value", axis=1)
y = fe_df.Value

In [5]:
# split into train and test set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# select num. and cat. features
num_features = X.select_dtypes(include="number").columns
cat_features = X.select_dtypes(exclude="number").columns

In [7]:
# create pipelines
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

preprocessor = create_preprocessor(num_features, cat_features)

pipelines = {
    "Dummy": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", DummyRegressor(strategy="mean"))
    ]),
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())]),
    "SVR": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", SVR(kernel="rbf"))]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=100, random_state=42,))]),
    "XG Boost": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(n_estimators=100, random_state=42))]),
}

In [8]:
# train the models
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.model_selection import cross_val_score
import joblib

results = {}

for model_name, pipeline in pipelines.items():
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    results[model_name] = {
        "CV Mean R²": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test R² score": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "MAEP": mean_absolute_percentage_error(y_test, y_pred),
    }
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [9]:
# save the metrics df
metrics_df = pd.DataFrame(results).T.round(2)
metrics_df.to_csv("../data/processed/metrics_df.csv", index=True)